# EstateAI: Production-Grade Real Estate Valuation Engine
### **Objective:** Implementing a premium AI pipeline with 50k+ samples, sub-location granularity, and ensemble stacking.

## 1. Audit Current Model Performance

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor, StackingRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_absolute_percentage_error
import joblib
import os

df = pd.read_csv('../data/house_data.csv')
print(f"Premium Dataset Loaded: {df.shape[0]} rows")
display(df.head())

## 2. Advanced Feature Engineering & Cleaning

In [ ]:
# Intelligent Ratios
df['house_age'] = 2025 - df['year_built']
df['total_rooms'] = df['bedrooms'] + df['bathrooms']
df['bhk_to_area_ratio'] = df['bedrooms'] / (df['area_sqft'] / 1000)
df['bath_per_bed'] = df['bathrooms'] / (df['bedrooms'] + 0.1)
df['luxury_flag'] = (df['area_sqft'] > 3500) | (df['property_type'].isin(['Villa', 'Penthouse']))
df['floor_efficiency'] = df['floors'] / (df['area_sqft'] / 1000)

# Premium Zones (Location specific)
premium_zones = ['Golf Course Road', 'DLF Phase 1', 'Saket', 'Vasant Kunj', 'Sector 150']
df['premium_zone'] = df['sub_location'].isin(premium_zones)

print("Feature Engineering Complete.")

## 3. Training Production-Grade Ensemble (Stacking)

In [ ]:
X = df.drop(['price'], axis=1)
y = np.log1p(df['price'])

cat_cols = ['location', 'sub_location', 'furnishing', 'property_type']
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
])

estimators = [
    ('xgb', XGBRegressor(n_estimators=300, max_depth=8, learning_rate=0.05)),
    ('lgbm', LGBMRegressor(n_estimators=300, max_depth=8, learning_rate=0.05))
]

stacking_model = StackingRegressor(estimators=estimators, final_estimator=RandomForestRegressor(n_estimators=100))

pipe = Pipeline([('pre', preprocessor), ('reg', stacking_model)])
pipe.fit(X, y)

os.makedirs('../model', exist_ok=True)
joblib.dump(pipe, '../model/house_model.pkl')
print("Production Stacking Ensemble Persistent.")

## 4. Confidence Score Logic
Based on sample density in sub-locations.

In [ ]:
def calculate_confidence(row, df):
    count = len(df[df['sub_location'] == row['sub_location']])
    if count > 1000: return 'High'
    elif count > 300: return 'Medium'
    else: return 'Low'